# 02b — Seed-focused Poincaré geodesics

Orthographic disk `(x,y)` on `S^2` with **Poincaré-disk geodesics** for every edge that touches at least one protein in `SEED_NODES`.

Use this notebook alone when you only need the seed neighborhood view. For stereo panels, random samples, and widgets, see `02_disk_views.ipynb`.

**Working directory:** `notebooks/dmercator/` (same as other D-Mercator notebooks).

Seed proteins are **annotated** on the disk (text labels in the plotting cell).


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.dpi"] = 200

import dmercator_io as dm

RUN_SUBDIR = "d2"
RUN_SUBDIR = os.environ.get("DMERCATOR_RUN", RUN_SUBDIR)
paths = dm.paths_for_run(RUN_SUBDIR)
_, df = dm.parse_inf_coord(paths["inf_coord"])
G = dm.load_edges_graph(paths["edge"])

xO, yO = dm.ortho_xy_disk(df)
pos_idx = {v: i for i, v in enumerate(df["Vertex"])}
print("RUN_SUBDIR:", RUN_SUBDIR, "|V|=", len(pos_idx), "|E|=", G.number_of_edges())

In [ ]:
# Poincaré disk ↔ upper half-plane, then H² geodesic; map back to disk coords


def _disk_to_upper(z: complex) -> complex:
    return 1j * (1 + z) / (1 - z)


def _upper_to_disk(w):
    return (w - 1j) / (w + 1j)


def _halfplane_geodesic_points(u1: complex, u2: complex, n: int):
    u1 = complex(u1)
    u2 = complex(u2)
    if abs(u1 - u2) < 1e-14:
        return np.array([u1], dtype=np.complex128)
    if abs(u1.real - u2.real) < 1e-12 * (1.0 + abs(u1.imag) + abs(u2.imag)):
        im = np.linspace(u1.imag, u2.imag, n)
        return u1.real + 1j * im
    denom = 2.0 * (u2.real - u1.real)
    c = (abs(u2) ** 2 - abs(u1) ** 2) / denom + 0j
    r = float(abs(u1 - c))
    t1 = np.angle(u1 - c)
    t2 = np.angle(u2 - c)
    delta = (t2 - t1 + np.pi) % (2 * np.pi) - np.pi
    angles = t1 + np.linspace(0.0, float(delta), n)
    return c + r * np.exp(1j * angles)


def poincare_geodesic_xy(z1, z2, n: int = 48):
    z1 = complex(z1)
    z2 = complex(z2)
    u1 = _disk_to_upper(z1)
    u2 = _disk_to_upper(z2)
    pts_h = _halfplane_geodesic_points(u1, u2, n)
    pts_d = _upper_to_disk(pts_h)
    return pts_d.real, pts_d.imag

**Seed list**

Edit ``SEED_NODES`` (exact protein IDs from the edgelist / ``Vertex`` column). Geodesics are drawn for every edge with **at least one** endpoint in the set.

- **Both** endpoints in ``SEED_NODES`` → `darkorange`
- **Exactly one** endpoint in ``SEED_NODES`` → `crimson`

Edges whose orthographic disk endpoint lies on the boundary (`|z| ≥ 0.999`) are skipped.

In [ ]:
from matplotlib.lines import Line2D

SEED_NODES = [
    "GRB2",
    "GIT1",
    "OCRL",
]

seeds_valid = [str(n).strip() for n in SEED_NODES if str(n).strip()]
missing = [n for n in seeds_valid if n not in pos_idx]
seeds_ok = [n for n in seeds_valid if n in pos_idx]
seed_set = set(seeds_ok)
if missing:
    print("Unknown or missing from coord table (skipped):", missing)
if not seeds_ok:
    raise ValueError("No valid seed names — fix SEED_NODES to match graph vertices.")

edge_rows = []
for a, b in G.edges():
    if a not in pos_idx or b not in pos_idx:
        continue
    a_in, b_in = a in seed_set, b in seed_set
    if not (a_in or b_in):
        continue
    edge_rows.append((a, b, a_in and b_in))

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(xO, yO, s=2, alpha=0.12, c="0.4")
col_single = "crimson"
col_both = "darkorange"
for a, b, both in edge_rows:
    ia, ib = pos_idx[a], pos_idx[b]
    z1 = complex(xO[ia], yO[ia])
    z2 = complex(xO[ib], yO[ib])
    if abs(z1) >= 0.999 or abs(z2) >= 0.999:
        continue
    gx, gy = poincare_geodesic_xy(z1, z2, n=48)
    ax.plot(gx, gy, lw=0.45, alpha=0.55, color=col_both if both else col_single)

ax.set_aspect("equal")
ax.add_patch(plt.Circle((0, 0), 1.0, fill=False, ls="--", lw=0.7))
ax.set_title("Poincaré geodesics — edges incident to " + ", ".join(seeds_ok))
ax.legend(
    handles=[
        Line2D([0], [0], color=col_single, lw=2, label="one seed endpoint"),
        Line2D([0], [0], color=col_both, lw=2, label="both endpoints are seeds"),
    ],
    loc="upper right",
    fontsize=8,
)
print("seed nodes:", seeds_ok)
print("edges drawn:", len(edge_rows))
for name in seeds_ok:
    j = pos_idx[name]
    ax.annotate(
        name,
        (xO[j], yO[j]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8,
        fontweight="bold",
        color="0.1",
        alpha=0.9,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.45", alpha=0.75),
    )

plt.tight_layout()
plt.show()